# EcoSmartHomes: DEAP Assessment Analysis
This notebook analyzes the DEAP PDF data to generate insights for the client's Retrofit Roadmap™.

In [ ]:
# Install required libraries for our charts (this might take a few seconds to run)
%pip install -q matplotlib pandas numpy

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# --- DATA EXTRACTED FROM DEAP PDF ---
house_data = {
    'address': 'Limerick City',
    'year_built': 2025,
    'total_area_m2': 213.15,
    'total_heat_loss_W_K': 220.93,
    'fabric_heat_loss_W_K': 132.02,
    'ventilation_heat_loss_W_K': 88.91,
    'annual_electricity_kWh': 5646,  # Total 6525 - 879 (wood pellets)
    'annual_secondary_heating_kWh': 879,
    'electricity_price_eur_per_kWh': 0.33,
    'wood_pellet_price_eur_per_kWh': 0.12
}
print("✅ Home Data Loaded Successfully!")

## 1. Heat Pump Readiness Test
To qualify for an SEAI heat pump grant, a home must have a Heat Loss Indicator (HLI) of **2.0 W/K m² or lower**.

In [ ]:
hli = house_data['total_heat_loss_W_K'] / house_data['total_area_m2']
print(f"Calculated Heat Loss Indicator (HLI): {hli:.2f} W/K m²")

if hli <= 2.0:
    print("✅ PASS: This home is Heat Pump Ready!")
else:
    print("❌ FAIL: This home needs insulation upgrades before installing a Heat Pump.")

## 2. Heat Loss Breakdown (Leakiness Score)
Visualizing exactly where the heat is escaping so we can target the right upgrades.

In [ ]:
labels = ['Fabric Loss (Walls, Roof, Floor)', 'Ventilation Loss (Drafts, Vents)']
sizes = [house_data['fabric_heat_loss_W_K'], house_data['ventilation_heat_loss_W_K']]
colors = ['#ff9999','#66b3ff']

fig, ax = plt.subplots(figsize=(7, 5))
ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
ax.axis('equal')  
plt.title('Where is the home losing heat?', fontsize=14, pad=20)
plt.show()

## 3. Carbon Tax Shield (10-Year Cost Projection)
Projecting the running costs over the next 10 years, assuming a conservative 4% annual increase in energy prices.

In [ ]:
years = np.arange(2025, 2035)
base_cost = (house_data['annual_electricity_kWh'] * house_data['electricity_price_eur_per_kWh']) + \
            (house_data['annual_secondary_heating_kWh'] * house_data['wood_pellet_price_eur_per_kWh'])

# Assume 4% annual energy inflation (carbon taxes + market rates)
inflation_rate = 0.04
projected_costs = [base_cost * ((1 + inflation_rate) ** i) for i in range(10)]
cumulative_costs = np.cumsum(projected_costs)

fig, ax1 = plt.subplots(figsize=(10, 6))

# Bar chart for annual cost
ax1.bar(years, projected_costs, color='#4CAF50', alpha=0.7, label='Annual Energy Bill')
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Annual Cost (€)', color='#4CAF50', fontsize=12)
ax1.tick_params(axis='y', labelcolor='#4CAF50')

# Line chart for cumulative cost
ax2 = ax1.twinx()
ax2.plot(years, cumulative_costs, color='#F44336', marker='o', linewidth=2, label='Cumulative Money Spent')
ax2.set_ylabel('Cumulative Cost (€)', color='#F44336', fontsize=12)
ax2.tick_params(axis='y', labelcolor='#F44336')

plt.title('10-Year Running Cost Projection (The Carbon Tax Shield)', fontsize=14, pad=15)
plt.grid(axis='y', linestyle='--', alpha=0.5)
fig.tight_layout()
plt.show()

print(f"Total projected spend over 10 years if nothing changes: €{cumulative_costs[-1]:,.2f}")